In [6]:
import pandas as pd

df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin-1')

print(f"✅ Rows loaded : {len(df):,}")
print(f"   Columns     : {len(df.columns)}")
df.head(2)

✅ Rows loaded : 180,519
   Columns     : 53


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class


In [7]:
cols_to_keep = {
    'order date (DateOrders)'       : 'shipment_date',
    'Order Item Quantity'           : 'unit_count',
    'Delivery Status'               : 'delivery_status',
    'Late_delivery_risk'            : 'late_risk_flag',
    'Days for shipping (real)'      : 'actual_ship_days',
    'Days for shipment (scheduled)' : 'scheduled_ship_days',
    'Category Name'                 : 'product_category',
    'Shipping Mode'                 : 'shipping_mode',
    'Order Region'                  : 'order_region',
    'Order Status'                  : 'order_status',
    'Sales'                         : 'sales_amount',
    'Order Profit Per Order'        : 'profit',
    'Type'                          : 'payment_type'
}

df_clean = df[list(cols_to_keep.keys())].rename(columns=cols_to_keep).copy()

df_clean['shipment_date'] = pd.to_datetime(
    df_clean['shipment_date'], errors='coerce'
).dt.strftime('%Y-%m-%d')

status_map = {
    'Shipping on time' : 'on_time',
    'Advance shipping' : 'on_time',
    'Late delivery'    : 'late',
    'Shipping canceled': 'cancelled'
}
df_clean['status'] = df_clean['delivery_status'].map(status_map).fillna('on_time')
df_clean = df_clean.dropna(subset=['shipment_date'])

print(f"✅ Clean rows : {len(df_clean):,}")
print(f"\nStatus breakdown:")
print(df_clean['status'].value_counts())
print(f"\nDate range   : {df_clean['shipment_date'].min()}  →  {df_clean['shipment_date'].max()}")

✅ Clean rows : 180,519

Status breakdown:
status
late         98977
on_time      73788
cancelled     7754
Name: count, dtype: int64

Date range   : 2015-01-01  →  2018-01-31


In [8]:
import numpy as np

shipments = pd.DataFrame()

shipments['shipment_date']    = df_clean['shipment_date']
shipments['unit_count']       = df_clean['unit_count']
shipments['product_category'] = df_clean['product_category']
shipments['shipping_mode']    = df_clean['shipping_mode']
shipments['status']           = df_clean['status']
shipments['late_risk_flag']   = df_clean['late_risk_flag']
shipments['sales_amount']     = df_clean['sales_amount'].fillna(0).round(2)
shipments['profit']           = df_clean['profit'].fillna(0).round(2)

shipments['carrier_id'] = df_clean['payment_type'].apply(
    lambda x: f"CAR_{(abs(hash(str(x))) % 24) + 1:03d}"
)

shipments['shipment_type'] = df_clean['order_status'].apply(
    lambda x: 'inbound' if str(x).upper() in ['COMPLETE','CLOSED','PROCESSING']
              else 'outbound'
)

shipments['pallet_count'] = df_clean['unit_count'].apply(
    lambda x: max(1, int(x) // 50)
)

shipments['scheduled_dock'] = df_clean['product_category'].apply(
    lambda x: f"D{(abs(hash(str(x))) % 12) + 1:02d}"
)

shipments = shipments[[
    'shipment_date', 'carrier_id', 'shipment_type',
    'product_category', 'unit_count', 'pallet_count',
    'scheduled_dock', 'shipping_mode', 'status',
    'late_risk_flag', 'sales_amount', 'profit'
]]

print(f"✅ Shipments table built : {len(shipments):,} rows")
print(f"\nColumn list:")
print(list(shipments.columns))
print(f"\nSample row:")
print(shipments.iloc[0])

✅ Shipments table built : 180,519 rows

Column list:
['shipment_date', 'carrier_id', 'shipment_type', 'product_category', 'unit_count', 'pallet_count', 'scheduled_dock', 'shipping_mode', 'status', 'late_risk_flag', 'sales_amount', 'profit']

Sample row:
shipment_date           2018-01-31
carrier_id                 CAR_002
shipment_type              inbound
product_category    Sporting Goods
unit_count                       1
pallet_count                     1
scheduled_dock                 D12
shipping_mode       Standard Class
status                     on_time
late_risk_flag                   0
sales_amount                327.75
profit                       91.25
Name: 0, dtype: object


In [9]:
import random
from datetime import datetime

random.seed(42)
np.random.seed(42)

START = datetime(2015, 1, 1)
END   = datetime(2018, 1, 31)
dates = pd.date_range(START, END, freq='D')

shifts_rows = []
shift_id = 1

for date in dates:
    for shift_type in ['morning', 'afternoon', 'night']:

        month    = date.month
        dow      = date.dayofweek

        seasonal = 1.35 if month == 12 else 1.2 if month == 11 \
                   else 1.1 if month in [3,9] else 0.85 if month in [6,7] else 1.0
        dow_mult = 1.25 if dow == 0 else 1.15 if dow == 4 \
                   else 0.7 if dow == 6 else 1.0
        s_mult   = 1.3 if shift_type == 'morning' \
                   else 1.0 if shift_type == 'afternoon' else 0.7

        planned  = int(np.clip(30 * seasonal * dow_mult * s_mult, 10, 60))
        absence  = np.random.choice([0,.05,.10,.15,.20], p=[.5,.25,.15,.07,.03])
        actual   = max(8, int(planned * (1 - absence)))
        labor    = round(actual * (8.0 + np.random.normal(0, 0.3)), 2)
        overtime = round(max(0, abs(np.random.normal(0, 0.5))), 2)
        sup      = f"SUP_{random.randint(1,8):02d}"

        shifts_rows.append([
            shift_id, date.strftime('%Y-%m-%d'), shift_type,
            planned, actual, labor, overtime, sup
        ])
        shift_id += 1

shifts = pd.DataFrame(shifts_rows, columns=[
    'shift_id', 'shift_date', 'shift_type', 'planned_workers',
    'actual_workers', 'labor_hours', 'overtime_hours', 'supervisor_id'
])

print(f"✅ Shifts generated : {len(shifts):,} rows")
print(f"\nShift type breakdown:")
print(shifts['shift_type'].value_counts())
print(f"\nAverage planned workers : {shifts['planned_workers'].mean():.1f}")
print(f"Average actual workers  : {shifts['actual_workers'].mean():.1f}")
print(f"\nSample:")
print(shifts.head(3).to_string())

✅ Shifts generated : 3,381 rows

Shift type breakdown:
shift_type
morning      1127
afternoon    1127
night        1127
Name: count, dtype: int64

Average planned workers : 31.2
Average actual workers  : 29.6

Sample:
   shift_id  shift_date shift_type  planned_workers  actual_workers  labor_hours  overtime_hours supervisor_id
0         1  2015-01-01    morning               39              39       298.99            0.16        SUP_02
1         2  2015-01-01  afternoon               30              30       254.21            0.38        SUP_01
2         3  2015-01-01      night               21              21       164.34            0.26        SUP_05


In [10]:
dock_rows = []
event_id  = 1

for _, s in shipments.iterrows():
    if s['status'] == 'cancelled':
        continue

    arr_h = random.randint(5, 20)
    arr_m = random.choice([0, 15, 30, 45])

    if s['status'] == 'no_show':
        dock_rows.append([
            event_id, s['shipment_date'], s['scheduled_dock'],
            s['carrier_id'],
            f"{arr_h:02d}:{arr_m:02d}", None, None, 0
        ])
    else:
        delay = 0 if s['status'] == 'on_time' else random.randint(20, 90)
        act_h = min(23, arr_h + delay // 60)
        act_m = (arr_m + delay) % 60
        dur   = random.randint(45, 180)
        end_h = min(23, act_h + dur // 60)
        end_m = (act_m + dur) % 60

        dock_rows.append([
            event_id, s['shipment_date'], s['scheduled_dock'],
            s['carrier_id'],
            f"{arr_h:02d}:{arr_m:02d}",
            f"{act_h:02d}:{act_m:02d}",
            f"{end_h:02d}:{end_m:02d}",
            delay
        ])
    event_id += 1

dock_events = pd.DataFrame(dock_rows, columns=[
    'event_id', 'event_date', 'dock_door', 'carrier_id',
    'scheduled_start', 'actual_start', 'actual_end', 'wait_time_min'
])

print(f"✅ Dock events generated : {len(dock_events):,} rows")
print(f"\nAvg wait time (late only) : {dock_events[dock_events['wait_time_min']>0]['wait_time_min'].mean():.1f} mins")
print(f"Dock doors used           : {dock_events['dock_door'].nunique()}")
print(f"\nSample:")
print(dock_events.head(3).to_string())

✅ Dock events generated : 172,765 rows

Avg wait time (late only) : 55.0 mins
Dock doors used           : 12

Sample:
   event_id  event_date dock_door carrier_id scheduled_start actual_start actual_end  wait_time_min
0         1  2018-01-31       D12    CAR_002           08:00        08:00      10:52              0
1         2  2018-01-13       D12    CAR_019           11:15        11:08      13:06             53
2         3  2018-01-13       D12    CAR_004           11:30        11:30      13:59              0


In [11]:
scan_rows = []
scan_id   = 1
shift_start_hour = {'morning': 6, 'afternoon': 14, 'night': 22}

for _, sh in shifts.iterrows():
    for h in range(8):
        fatigue  = 1.0 - h * 0.015
        units    = max(0, int(
            sh['actual_workers'] * 22 * fatigue *
            np.random.normal(1.0, 0.12)
        ))
        scan_type = random.choice(['receive','sort','load','dispatch'])
        start_h   = shift_start_hour[sh['shift_type']]
        scan_dt   = f"{sh['shift_date']} {(start_h + h) % 24:02d}:00:00"

        scan_rows.append([scan_id, scan_dt, sh['shift_id'], units, scan_type])
        scan_id += 1

throughput_scans = pd.DataFrame(scan_rows, columns=[
    'scan_id', 'scan_datetime', 'shift_id',
    'units_scanned', 'scan_type'
])

print(f"✅ Throughput scans generated : {len(throughput_scans):,} rows")
print(f"\nScan type breakdown:")
print(throughput_scans['scan_type'].value_counts())
print(f"\nAvg units per scan : {throughput_scans['units_scanned'].mean():.1f}")
print(f"Max units per scan : {throughput_scans['units_scanned'].max()}")

✅ Throughput scans generated : 27,048 rows

Scan type breakdown:
scan_type
receive     6842
dispatch    6756
load        6755
sort        6695
Name: count, dtype: int64

Avg units per scan : 617.2
Max units per scan : 1638


In [12]:
d_rows  = []
d_id    = 1

d_types = ['carrier_no_show', 'equipment_failure',
           'weather_delay', 'volume_surge', 'staff_shortage']

d_cause = {
    'carrier_no_show'  : 'Carrier did not arrive',
    'equipment_failure': 'Forklift or conveyor down',
    'weather_delay'    : 'Weather-related delay',
    'volume_surge'     : 'Unexpected volume spike',
    'staff_shortage'   : 'Attendance below threshold'
}

morning_shifts = shifts[shifts['shift_type'] == 'morning'].copy()
morning_shifts = morning_shifts.set_index('shift_date')['shift_id']

for date in dates:
    month = date.month
    prob  = 0.18 if month in [11, 12, 1, 2] else 0.10

    if random.random() < prob:
        dt       = random.choice(d_types)
        sev      = random.choices(
                       ['low','medium','high','critical'],
                       weights=[.35, .40, .18, .07]
                   )[0]
        dur      = round(random.uniform(0.5, 8.0), 1)
        cost     = round(dur * random.uniform(800, 3500), 2)
        date_str = date.strftime('%Y-%m-%d')
        s_ref    = int(morning_shifts.get(date_str, 1))

        d_rows.append([
            d_id, date_str, dt, sev,
            dur, d_cause[dt], cost, s_ref
        ])
        d_id += 1

disruptions = pd.DataFrame(d_rows, columns=[
    'disruption_id', 'disruption_date', 'disruption_type',
    'severity', 'duration_hours', 'root_cause',
    'cost_impact', 'shift_id'
])

print(f"✅ Disruptions generated : {len(disruptions):,} rows")
print(f"\nType breakdown:")
print(disruptions['disruption_type'].value_counts())
print(f"\nSeverity breakdown:")
print(disruptions['severity'].value_counts())
print(f"\nTotal cost impact : ${disruptions['cost_impact'].sum():,.2f}")

✅ Disruptions generated : 131 rows

Type breakdown:
disruption_type
equipment_failure    34
carrier_no_show      30
weather_delay        28
volume_surge         25
staff_shortage       14
Name: count, dtype: int64

Severity breakdown:
severity
low         51
medium      50
high        24
critical     6
Name: count, dtype: int64

Total cost impact : $1,133,166.14


In [14]:
shipments.to_csv('shipments.csv',               index=False)
shifts.to_csv('shifts.csv',                     index=False)
dock_events.to_csv('dock_events.csv',           index=False)
throughput_scans.to_csv('throughput_scans.csv', index=False)
disruptions.to_csv('disruptions.csv',           index=False)

print("✅ All 5 files exported successfully")
print(f"\n   shipments.csv         — {len(shipments):,} rows")
print(f"   shifts.csv            — {len(shifts):,} rows")
print(f"   dock_events.csv       — {len(dock_events):,} rows")
print(f"   throughput_scans.csv  — {len(throughput_scans):,} rows")
print(f"   disruptions.csv       — {len(disruptions):,} rows")

✅ All 5 files exported successfully

   shipments.csv         — 180,519 rows
   shifts.csv            — 3,381 rows
   dock_events.csv       — 172,765 rows
   throughput_scans.csv  — 27,048 rows
   disruptions.csv       — 131 rows


In [15]:
from google.colab import files

files.download('shipments.csv')
files.download('shifts.csv')
files.download('dock_events.csv')
files.download('throughput_scans.csv')
files.download('disruptions.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>